To do:
- [X] Add future price values to state representation
- [X] Agent that looks at spot price and optimizes over the next day
- [X] Add reinforcement learning agents?
- [X] Add plots


- [x] Add local solar generation?
- [x] Add local wind generation?

- [x] Realistic battery parameters, source and price?

Self discharge of battery is 0.002% per hour so maybe not relevant
- [x] Reduction max capacity over time?

# Fixing the demand and supply mismatch in the danish energy system

The Danish energy system is facing a significant challenge in balancing supply and demand, especially with the increasing integration of renewable energy sources. This leads to prices that can fluctuate wildly.


## Supply side
More stable supply like nuclear power.

One potential solution to this issue is the implementation of large-scale batteries on a structural level. Like hydro power plants, these batteries can store excess energy generated during periods of low demand and release it during peak demand times. This would help to stabilize the grid and reduce price volatility. This could in the future make the energy prices more stable, but it would require significant investment and infrastructure development. 

(Making the supply curve more flat)


## Demand side
With the current system consumers are incentivized to use energy during off-peak hours when prices are lower, which can help to balance supply and demand. However, this is not a perfect solution, as it relies on consumer behavior and may not be sufficient to address the underlying issues in the energy system.

Another potential solution is for consumers to use energy storage systems, such as home batteries, to store energy when prices are low and use it when prices are high. This can help to reduce demand during peak hours and provide a more stable energy supply. However, this also requires significant investment from consumers and may not be feasible for everyone.

(Making the demand more responsive to price)


# Simulation Environment

We are going to create a simulation environment to model the energy market for a single consumer. 
The consumer is small enough to not affect the market price, as this would otherwise make the problem more complex. 
It takes the market price $p_t$ as given.

We also assume that the consumer knows their own energy consumption patterns in advance and it is $d_t$ for each time period $t$. 
For many consumers, this is a reasonable assumption, as much of energy consumption is predictable (e.g., heating, cooling, appliances). 
Some consumers may have more variable consumption patterns, but this would require a more complex model to capture the uncertainty in demand.

We assume that the demand must be met at all times, meaning that the consumer cannot simply choose to not consume energy during peak hours.
This is a reasonable assumption, as most consumers have essential energy needs that cannot be easily shifted (e.g., heating, cooling, lighting), and if they could be shifted, you could simply shift them to off-peak hours.

The consumer buys $e_t$ kWh of energy from the market at time $t$.

The consumer has a battery of size X kWh.
The battery has a maximum charge and discharge rate of Y kW per hour.
The battery has an efficiency of Z%, meaning that if you charge the battery with 1 kWh, you can only get Z kWh out of it.

The price of the battery should be investigated, as it may not be cost-effective for all consumers to invest in a battery.


Degradation of the battery should also be taken into account, as it may affect the long-term cost-effectiveness of the battery.
Lifetime of the battery should also be considered, as it may affect the long-term cost-effectiveness of the battery.


## Environment


## Agent
Represents a consumer who wants to minimize their energy costs by deciding how much energy to buy from the market and how much to store in the battery.
Takes in state and outputs an action.

## State
Battery level, $b_t$
Market prices, $p_t$
If some demand is unpredictable, we could also include this $d^{\text{unpredictable}}_t$, in the state.
Other factors that could be included in the state to help the agent predict future market prices like weather conditions or economic indicators.

## Action
Each hour decide how much energy to buy, $e_t$.
If bigger than demand, the excess energy is stored in the battery.
If smaller than demand, the deficit is covered by the battery.


## Dynamics
If $e_t > d_t$, then the excess energy is stored in the battery, and the battery level increases by $(e_t - d_t) * Z\%$.
$$b_{t+1} = b_t + (e_t - d_t) * Z\% $$
If $e_t < d_t$, then the deficit is covered by the battery, and the battery level decreases by $d_t - e_t$.
$$b_{t+1} = b_t - (d_t - e_t)$$

The other factors market prices, unpredictable demand, weather conditions, and economic indicators are exogenous and can be modeled as stochastic processes or based on historical data. 

## Feasibility
$0 \leq b_t \leq X$ (battery capacity constraint)
$abs(e_t - d_t) \leq Y$ (charge/discharge rate constraint)



## Reward (cost)
p_t * e_t

## Episode
each day the new markets prices are revealed
each hour new others factors are revealed (e.g., weather conditions, economic indicators)

In [ ]:
import numpy as np
import pandas as pd
from pyomo.environ import *
# import deepcopy
from copy import deepcopy

In [ ]:
class State:
    def __init__(self, time_step, battery_level, energy_price, carbon_intensity, other_factors=None):
        self.time_step = time_step
        self.battery_level = battery_level
        self.energy_price = energy_price
        self.carbon_intensity = carbon_intensity
        self.other_factors = other_factors


class Action:
    def __init__(self, buy_energy_amount):
        self.buy_energy_amount = buy_energy_amount


class Cost: 
    def __init__(self, amount, carbon_emissions, green_factor):
        self.amount = amount
        self.carbon_emissions = carbon_emissions
        self.weighted_cost = (1 - green_factor) * amount + green_factor * carbon_emissions

class Parameters:
    def __init__(self, initial_battery_level, max_battery_capacity, max_charging_rate, max_discharging_rate, charge_efficiency, discharge_efficiency, green_factor):
        self.initial_battery_level = initial_battery_level
        self.max_battery_capacity = max_battery_capacity
        self.max_charging_rate = max_charging_rate
        self.max_discharging_rate = max_discharging_rate
        self.charge_efficiency = charge_efficiency
        self.discharge_efficiency = discharge_efficiency
        self.green_factor = green_factor


class ExogenousValues:
    def __init__(self, energy_price, carbon_intensity, other_factors=None):
        self.energy_price = energy_price
        self.carbon_intensity = carbon_intensity
        self.other_factors = other_factors


class SimulationResult:
    def __init__(self):
        self.states = []
        self.actions = []
        self.costs = []

    def get_total_cost(self, cost_type):
        return sum(getattr(cost, cost_type) for cost in self.costs)
    
    def is_similar_to(self, other_simulation_result, tolerance=1e-6):
        for t in range(len(self.states)):
            assert abs(self.states[t].battery_level - other_simulation_result.states[t].battery_level) <= tolerance, f"States differ at time step {t}: {self.states[t].battery_level} vs {other_simulation_result.states[t].battery_level}"
            assert abs(self.states[t].energy_price - other_simulation_result.states[t].energy_price) <= tolerance, f"States differ at time step {t}: {self.states[t].energy_price} vs {other_simulation_result.states[t].energy_price}"
            #assert abs(self.states[t].carbon_intensity - other_simulation_result.states[t].carbon_intensity) <= tolerance, f"States differ at time step {t}: {self.states[t].carbon_intensity} vs {other_simulation_result.states[t].carbon_intensity}"

        for t in range(len(self.actions)):
            assert abs(self.actions[t].buy_energy_amount - other_simulation_result.actions[t].buy_energy_amount) <= tolerance, f"Actions differ at time step {t}: {self.actions[t].buy_energy_amount} vs {other_simulation_result.actions[t].buy_energy_amount}"

        for t in range(len(self.costs)):
            assert abs(self.costs[t].amount - other_simulation_result.costs[t].amount) <= tolerance, f"Costs differ at time step {t}: {self.costs[t].amount} vs {other_simulation_result.costs[t].amount}"
            #assert abs(self.costs[t].carbon_emissions - other_simulation_result.costs[t].carbon_emissions) <= tolerance, f"Costs differ at time step {t}: {self.costs[t].carbon_emissions} vs {other_simulation_result.costs[t].carbon_emissions}"
            assert abs(self.costs[t].weighted_cost - other_simulation_result.costs[t].weighted_cost) <= tolerance, f"Costs differ at time step {t}: {self.costs[t].weighted_cost} vs {other_simulation_result.costs[t].weighted_cost}"
            
        

class EvaluationResult:
    def __init__(self, policy_name):
        self.policy_name = policy_name
        self.simulation_results = []

    def print_average_costs(self):
        for cost_type in ['amount', 'carbon_emissions', 'weighted_cost']:
            avg_cost = sum(sim.get_total_cost(cost_type) for sim in self.simulation_results) / len(self.simulation_results)
            print(f"Average {cost_type}: {avg_cost:.2f}")
        


class Environment:
    def __init__(self, parameters, demand_profile, epsilon=1e-4):
        self.parameters = parameters
        self.demand_profile = demand_profile
        self.epsilon = epsilon


    def check_and_fix_action(self, state, action):
        # return True if action is feasible in the given state, else False

        if action.buy_energy_amount > self.demand_profile[state.time_step]:  # Charging
            lowest_max_limit = min(self.demand_profile[state.time_step] + self.parameters.max_charging_rate,
                               self.demand_profile[state.time_step] + (self.parameters.max_battery_capacity - state.battery_level) / self.parameters.charge_efficiency)
            
            if action.buy_energy_amount > lowest_max_limit + self.epsilon:
                print("Action exceeds limits for charging.")
                print(f"Action buy energy amount: {action.buy_energy_amount} kWh")
                print(f"Demand at time step {state.time_step}: {self.demand_profile[state.time_step]} kWh")
                print(f"Lowest limit for charging: {lowest_max_limit} kWh")
                print(f"Battery level: {state.battery_level} kWh")
                
                raise ValueError("Action exceeds limits for charging.")
            elif action.buy_energy_amount > lowest_max_limit:
                action.buy_energy_amount = lowest_max_limit

        else:  # Discharging
            highest_min_limit = max(self.demand_profile[state.time_step] - self.parameters.max_discharging_rate * self.parameters.discharge_efficiency,
                                    self.demand_profile[state.time_step] - state.battery_level * self.parameters.discharge_efficiency)


            if  action.buy_energy_amount < highest_min_limit - self.epsilon:
                print("Action exceeds limits for discharging.")
                print(f"Action buy energy amount: {action.buy_energy_amount} kWh")
                print(f"Demand at time step {state.time_step}: {self.demand_profile[state.time_step]} kWh")
                print(f"Highest limit for discharging: {highest_min_limit} kWh")
                print(f"Battery level: {state.battery_level} kWh")
                
                raise ValueError("Action exceeds limits for discharging.")
            elif action.buy_energy_amount < highest_min_limit:
                action.buy_energy_amount = highest_min_limit

        return action

    def get_cost(self, state, action):
        # return cost of taking action in the given state
        energy_cost_amount = action.buy_energy_amount * state.energy_price
        carbon_emissions = action.buy_energy_amount * state.carbon_intensity
        return Cost(amount=energy_cost_amount, carbon_emissions=carbon_emissions, green_factor=self.parameters.green_factor)

    def transition(self, state, action, next_exogen_values):
        # return next state
        excess_energy = action.buy_energy_amount - self.demand_profile[state.time_step]

        # update battery level based on action
        if excess_energy > 0:  # Charging
            new_battery_level = state.battery_level + excess_energy * self.parameters.charge_efficiency
        else:  # Discharging
            new_battery_level = state.battery_level + excess_energy / self.parameters.discharge_efficiency  # excess_energy is negative for discharging

        # get next exogen values
        return State(time_step=state.time_step + 1,
                     battery_level=new_battery_level,
                     energy_price=next_exogen_values.energy_price,
                     carbon_intensity=next_exogen_values.carbon_intensity,
                     other_factors=next_exogen_values.other_factors)
    
    def do_simulation(self, policy, exogen_data):
        result = SimulationResult()

        state = State(time_step=0,
                        battery_level=self.parameters.initial_battery_level,
                        energy_price=exogen_data[0].energy_price,
                        carbon_intensity=exogen_data[0].carbon_intensity,
                        other_factors=exogen_data[0].other_factors)

        for t in range(len(exogen_data)):
            result.states.append(state)

            policy_action = policy.select_action(state)
            policy_action = self.check_and_fix_action(state, policy_action)
            
            result.actions.append(policy_action)
            
            
            cost = self.get_cost(state, policy_action)
            result.costs.append(cost)

            if t < len(exogen_data) - 1:  # No need to transition on the last step
                next_exogen_values = exogen_data[t + 1]
                state = self.transition(state, policy_action, next_exogen_values)

        return result
        

    def evaluate_policies(self, policies, all_exogen_data):

        all_results = []
        for policy in policies:
            policy_name = policy.__name__
            print(f"Evaluating policy: {policy_name}")
            result = EvaluationResult(policy_name)
        
            for sim in range(len(all_exogen_data)):
                policy_instance = policy(self)
                print(f"  Simulation {sim + 1}/{len(all_exogen_data)}")
                exogen_data = all_exogen_data[sim]

                if policy_instance.is_hindsight_policy:
                    policy_instance.see_future_exogen_values(exogen_data)

                simulation_result = self.do_simulation(policy_instance, exogen_data)
                result.simulation_results.append(simulation_result)

                if policy_instance.expected_simulation_result is not None:
                    # check that hindsight policy actions are the same as optimal actions
                    simulation_result.is_similar_to(policy_instance.expected_simulation_result, tolerance=self.epsilon)

            print(f"Policy {policy_name} evaluation completed.")
            result.print_average_costs()
            
            all_results.append(result)

        return all_results


        
        
    

class Agent:
    def __init__(self, environment):
        self.environment = environment
        self.is_hindsight_policy = False  # Set to True for hindsight agents that can see future exogenous values
        self.expected_simulation_result = None  # For hindsight agents to store the expected optimal simulation result based on the exogenous data they see

    def see_future_exogen_values(self, exogen_data):
        raise NotImplementedError("This method should be implemented by hindsight agents that can see future exogenous values")

    def select_action(self, state):
        # return action based on the current state
        raise NotImplementedError("This method should be implemented by subclasses")
    

In [ ]:
class DummyAgent(Agent):
    def select_action(self, state):
        return Action(buy_energy_amount=self.environment.demand_profile[state.time_step])
    

In [ ]:
def solve_MILP_model(exogen_data, parameters, demand_profile):
    model = ConcreteModel()
    # Sets
    time_steps = range(len(exogen_data))
    model.T = Set(initialize=time_steps)

    # Parameters
    model.demand = Param(model.T, initialize={t: demand_profile[t] for t in model.T})
    model.energy_price = Param(model.T, initialize={t: exogen_data[t].energy_price for t in model.T})
    model.carbon_intensity = Param(model.T, initialize={t: exogen_data[t].carbon_intensity for t in model.T})

    model.max_battery_capacity = Param(initialize=parameters.max_battery_capacity)
    model.max_charging_rate = Param(initialize=parameters.max_charging_rate)
    model.max_discharging_rate = Param(initialize=parameters.max_discharging_rate)
    model.charge_efficiency = Param(initialize=parameters.charge_efficiency)
    model.discharge_efficiency = Param(initialize=parameters.discharge_efficiency)
    model.green_factor = Param(initialize=parameters.green_factor)


    # Variables
    model.buy_energy = Var(model.T, within=NonNegativeReals)
    model.charge_battery = Var(model.T, within=NonNegativeReals)
    model.discharge_battery = Var(model.T, within=NonNegativeReals)
    model.battery_level = Var(model.T, within=NonNegativeReals)
    model.charge = Var(model.T, within=Binary)  # 1 if charging, 0 if discharging

    # Objective
    def objective_rule(m):
        return sum(
            (1 - m.green_factor) * m.buy_energy[t] * m.energy_price[t] + 
            m.green_factor * m.buy_energy[t] * m.carbon_intensity[t]
            for t in m.T
        )
    model.objective = Objective(rule=objective_rule, sense=minimize)

    # Constraints
    # Demand must be met at each time step
    def demand_constraint_rule(m, t):
        return m.buy_energy[t] + m.discharge_battery[t] * m.discharge_efficiency == m.demand[t] + m.charge_battery[t]
    
    model.demand_constraint = Constraint(model.T, rule=demand_constraint_rule)

    # Battery level constraints
    def battery_level_rule(m, t):
        return m.battery_level[t] <= m.max_battery_capacity

    model.battery_capacity_constraint = Constraint(model.T, rule=battery_level_rule)

    # Max charging and discharging rates
    def charging_rate_rule(m, t):
        return m.charge_battery[t] <= m.max_charging_rate * m.charge[t]
    model.charging_rate_constraint = Constraint(model.T, rule=charging_rate_rule)

    def discharging_rate_rule(m, t):
        return m.discharge_battery[t] <= m.max_discharging_rate * (1 - m.charge[t])
    model.discharging_rate_constraint = Constraint(model.T, rule=discharging_rate_rule)

    def discharging_less_than_battery_rule(m, t):
        return m.discharge_battery[t] <= m.battery_level[t]
    
    model.discharging_battery_constraint = Constraint(model.T, rule=discharging_less_than_battery_rule)

    # Battery level dynamics
    def battery_dynamics_rule(m, t):
        if t == 0:
            return m.battery_level[t] == parameters.initial_battery_level
        else:
            return m.battery_level[t] == m.battery_level[t-1] + m.charge_battery[t-1] * m.charge_efficiency - m.discharge_battery[t-1]
    model.battery_dynamics_constraint = Constraint(model.T, rule=battery_dynamics_rule)
    
    # Solve the model
    solver = SolverFactory('gurobi')
    if not solver.available():
        raise RuntimeError("Gurobi solver is not available in your environment.")
    results = solver.solve(model, tee=False)
    status = results.Solver()['Termination condition'].value
    assert status == 'optimal', f'error occurred, status: {status}.  Check model!'
    return model
    

In [ ]:
class OptimalHindsightAgent(Agent):
    def __init__(self, environment):
        super().__init__(environment)
        self.is_hindsight_policy = True
        self.expected_simulation_result = SimulationResult()

    def see_future_exogen_values(self, exogen_data):
        
        model = solve_MILP_model(exogen_data, self.environment.parameters, self.environment.demand_profile)

        # save optimal actions for each time step
        self.expected_simulation_result.states = [State(time_step=t,
                                                        battery_level=model.battery_level[t].value,
                                                        energy_price=model.energy_price[t],
                                                        carbon_intensity=model.carbon_intensity[t]) for t in model.T]
        self.expected_simulation_result.actions = [Action(buy_energy_amount=model.buy_energy[t].value) for t in model.T]
        self.expected_simulation_result.costs = [Cost(amount=model.buy_energy[t].value * model.energy_price[t],
                                                        carbon_emissions=model.buy_energy[t].value * model.carbon_intensity[t],
                                                        green_factor=self.environment.parameters.green_factor) for t in model.T]
            
    def select_action(self, state):
        return self.expected_simulation_result.actions[state.time_step]

In [ ]:
class OptimizeOverNextDayAgent(Agent):
    def __init__(self, environment):
        super().__init__(environment)
        assert environment.parameters.green_factor == 0, "OptimizeOverNextDayAgent is only implemented for cost minimization (green_factor=0)"
        self.expected_simulation_result = SimulationResult()
        self.saved_exogen_data = {}


    def select_action(self, state):
        
        any_future_prices = False

        for t in state.other_factors['future_prices']:
            any_future_prices = True
            self.saved_exogen_data[t] = ExogenousValues(
                energy_price=state.other_factors['future_prices'][t],
                carbon_intensity=0,  # Carbon intensity is not relevant for this agent since green_factor=0
                other_factors={}
            )

        if any_future_prices:
            parameters = deepcopy(self.environment.parameters)
            parameters.initial_battery_level = state.battery_level

            t = state.time_step
            exogen_data = []
            while t in self.saved_exogen_data:
                exogen_data.append(self.saved_exogen_data[t])
                t += 1

            demand_profile = self.environment.demand_profile[state.time_step:state.time_step + len(exogen_data)]

            model = solve_MILP_model(exogen_data, parameters, demand_profile)
            
            # save states, actions, and costs for the optimized period
            for t in range(state.time_step, state.time_step + len(exogen_data)):
                new_state = State(time_step=t,
                              battery_level=model.battery_level[t - state.time_step].value,
                              energy_price=model.energy_price[t - state.time_step],
                              carbon_intensity=model.carbon_intensity[t - state.time_step])
                action = Action(buy_energy_amount=model.buy_energy[t - state.time_step].value)
                cost = Cost(amount=model.buy_energy[t - state.time_step].value * model.energy_price[t - state.time_step],
                            carbon_emissions=model.buy_energy[t - state.time_step].value * model.carbon_intensity[t - state.time_step],
                            green_factor=0)
                if t < len(self.expected_simulation_result.states):
                    self.expected_simulation_result.states[t] = new_state
                    self.expected_simulation_result.actions[t] = action
                    self.expected_simulation_result.costs[t] = cost
                else:
                    self.expected_simulation_result.states.append(new_state)
                    self.expected_simulation_result.actions.append(action)
                    self.expected_simulation_result.costs.append(cost)

        return self.expected_simulation_result.actions[state.time_step]
        
    

In [ ]:
class OptimizeOverNextWeekAgent(Agent):
    def select_action(self, state):
        return super().select_action(state)

In [ ]:
parameters = Parameters(
    initial_battery_level=0, # Initial battery level in kWh
    max_battery_capacity=1000, # Maximum battery capacity in kWh
    max_charging_rate=500, # Maximum charge rate in kW
    charge_efficiency=0.95, # Efficiency of charging (0 to 1)
    max_discharging_rate=500, # Maximum discharge rate in kW
    discharge_efficiency=0.95, # Efficiency of discharging (0 to 1)
    green_factor=0 # Weighting factor for carbon intensity in cost calculation (0 to 1)
)




Demand data from https://www.kaggle.com/datasets/csafrit2/steel-industry-energy-consumption

In [ ]:
# demand profile

demand_df = pd.read_csv("steel_industry_hourly_usage.csv")


# get weekday from date column and add as new column
demand_df['date'] = pd.to_datetime(demand_df['date'])
demand_df['week'] = demand_df['date'].dt.isocalendar().week
demand_df['day_of_week'] = demand_df['date'].dt.weekday
demand_df['hour_of_day'] = demand_df['date'].dt.hour

# remove duplicate rows in week and day of week and hour of day
demand_df = demand_df.drop_duplicates(subset=['week', 'day_of_week', 'hour_of_day'])


In [ ]:
df = pd.read_csv("final_data.csv")

df['datetime'] = pd.to_datetime(df['datetime'])

df['week'] = df['datetime'].dt.isocalendar().week

# merge df with demand df on week, day_of_week and hour_of_day
print(len(df))
df = pd.merge(df, demand_df, how='left', on=['week', 'day_of_week', 'hour_of_day'])
print(len(df))

print(df[df['Usage_kWh'].isnull()])


In [ ]:

all_exogen_data = []
all_demands = []

for zone in ['DK-DK1', 'DK-DK2']:
    zone_data = df[df['zone'] == zone].sort_values('datetime')

    # add index column to zone data
    zone_data = zone_data.reset_index(drop=True)


    zone_exogen_data = []
    zone_demands = []

    for i, row in zone_data.iterrows():
        future_prices = {}

        row_hour = row['hour_of_day']

        if i == 0:
            # add prices for this day
            day_data = zone_data[zone_data['datetime'].dt.date == row['datetime'].date()]
            future_prices = {j: future_row['value_spot']/1000 for j, future_row in day_data.iterrows()}
            if row_hour >= 13: # also add prices for next day¨
                next_day_data = zone_data[zone_data['datetime'].dt.date == row['datetime'].date() + pd.Timedelta(days=1)]
                future_prices.update({j: future_row['value_spot']/1000 for j, future_row in next_day_data.iterrows()})

        
        elif row_hour == 13: # if its 1 pm or after also add prices for the next day     
            next_day_data = zone_data[zone_data['datetime'].dt.date == row['datetime'].date() + pd.Timedelta(days=1)]
            future_prices = {j: future_row['value_spot']/1000 for j, future_row in next_day_data.iterrows()}

        zone_exogen_data.append(ExogenousValues(
            energy_price=row['value_spot']/1000,  # Convert to EUR/kWh
            carbon_intensity=row['carbonIntensity']/1000,  # Convert to kgCO2/kWh
            other_factors={'future_prices': future_prices}
        ))
        zone_demands.append(row['Usage_kWh'])
    
    all_exogen_data.append(zone_exogen_data)
    all_demands.append(zone_demands)


In [ ]:
print(len(all_exogen_data[0]))

In [ ]:
environment = Environment(parameters, all_demands[0])

In [ ]:
policies = []

policies.append(DummyAgent)
policies.append(OptimizeOverNextDayAgent)
policies.append(OptimalHindsightAgent)

all_results = environment.evaluate_policies(policies, all_exogen_data)